# Per-Shot Correlation Analysis: Logical Error Distribution vs Fixed-Class LLRs

This notebook focuses on **per-shot correlation analysis** to understand whether the logical error distribution predicts which error class will have the lowest fixed-class LLR *within each shot*.

Key metrics:
1. **Per-shot Spearman correlation**: Does rank predict LLR ordering within each shot?
2. **Optimal selection rate**: How often does the highest-probability error have the lowest LLR?
3. **Regret analysis**: How much LLR penalty for using distribution-based selection?
4. **Rank-stratified success rate**: Do top-k errors contain the winner more often than random?

## 1. Data Collection Configuration

Configure the simulation parameters below, then run the cell to generate the script command.

In [ ]:
# ==============================================================================
# CONFIGURATION - Set your simulation parameters here
# ==============================================================================

# Code parameters
N_QUBITS = 144          # BB code variant: 72, 90, 108, 144, 288, 360, 756
P = 0.003               # Physical error rate

# Simulation parameters
N_SHOTS = 1000          # Number of shots to simulate
N_JOBS = 126             # Parallel jobs (-1 for all CPUs)
SEED = 42               # Random seed

# Error selection: Choose ONE of the following methods
# Method 1: Range-based selection (set RANK_RANGE, leave N_ERRORS as None)
# Method 2: Uniform sampling (set N_ERRORS, leave RANK_RANGE as None)

RANK_RANGE = (0, 1000, 100)  # (start, end, step) - selects ranks 0, 10, 20, ..., 90
# RANK_RANGE = (0, 50, 1)  # Top 50 consecutive errors
# RANK_RANGE = (0, 20, 1)  # Top 20 consecutive errors
# RANK_RANGE = None        # Uncomment to use N_ERRORS instead

N_ERRORS = None            # Number of uniformly spaced errors (use if RANK_RANGE is None)
# N_ERRORS = 10            # Uncomment to sample 10 errors uniformly across distribution

# Output path (auto-generated based on parameters)
OUTPUT_DIR = "simulations/data/correlation_analysis"

In [ ]:
# ==============================================================================
# Generate script command based on configuration
# ==============================================================================

from pathlib import Path

# Validate configuration
if RANK_RANGE is not None and N_ERRORS is not None:
    raise ValueError("Set only ONE of RANK_RANGE or N_ERRORS, not both")
if RANK_RANGE is None and N_ERRORS is None:
    raise ValueError("Set either RANK_RANGE or N_ERRORS")

# Generate output filename
if RANK_RANGE is not None:
    start, end, step = RANK_RANGE
    selection_desc = f"rank{start}-{end}_step{step}"
    error_selection_arg = f"--rank-range {start} {end} {step}"
else:
    selection_desc = f"uniform{N_ERRORS}"
    error_selection_arg = f"--n-errors {N_ERRORS}"

OUTPUT_FILENAME = f"bb_n{N_QUBITS}_p{P}_{selection_desc}_shots{N_SHOTS}.parquet"
OUTPUT_PATH = Path(OUTPUT_DIR) / OUTPUT_FILENAME

# Build command
SCRIPT_PATH = "simulations/analysis/scripts/run_distribution_correlation_analysis.py"

cmd_parts = [
    f"python {SCRIPT_PATH}",
    f"    --n-qubits {N_QUBITS}",
    f"    --p {P}",
    f"    --n-shots {N_SHOTS}",
    f"    --n-jobs {N_JOBS}",
    f"    --seed {SEED}",
    f"    {error_selection_arg}",
    f"    --output {OUTPUT_PATH}",
]

command = " \\\n".join(cmd_parts)

# Display
print("=" * 70)
print("GENERATED COMMAND")
print("=" * 70)
print()
print(command)
print()
print("=" * 70)
print(f"Output file: {OUTPUT_PATH}")
print("=" * 70)

# Store for later use
RESULTS_PATH = Path("../../..") / OUTPUT_PATH

In [ ]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
from scipy import stats

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

## 2. Load Results

In [ ]:
# ==============================================================================
# Load results (uses RESULTS_PATH from configuration above)
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first using the command above.")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame
df_results = pd.read_parquet(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    if isinstance(value, list) and len(value) > 5:
        print(f"  {key}: [{value[0]}, {value[1]}, ..., {value[-1]}] (len={len(value)})")
    elif isinstance(value, dict):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nDataFrame preview:")
df_results.head(10)

## 3. Per-Shot Correlation Analysis

In [ ]:
# ==============================================================================
# Compute Per-Shot Statistics
# ==============================================================================

per_shot_stats = []

for shot_id, group in df_results.groupby("shot_id"):
    # Sort by error rank to ensure consistent ordering
    group = group.sort_values("error_rank")
    
    # Per-shot Spearman correlation (rank vs LLR)
    rho, p_val = stats.spearmanr(group["error_rank"], group["fixed_llr"])
    
    # Find which rank achieved the minimum LLR
    min_idx = group["fixed_llr"].idxmin()
    min_llr_rank = group.loc[min_idx, "error_rank"]
    min_llr = group["fixed_llr"].min()
    
    # Rank-0 statistics (lowest rank in this selection)
    lowest_rank = group["error_rank"].min()
    lowest_rank_llr = group[group["error_rank"] == lowest_rank]["fixed_llr"].values[0]
    lowest_rank_is_best = (min_llr_rank == lowest_rank)
    
    # Regret: LLR penalty for using lowest rank instead of optimal
    regret = lowest_rank_llr - min_llr
    
    # Baseline difficulty (best_llr from standard decoding)
    best_llr = group["best_llr"].iloc[0]
    
    # Mean LLR for random selection baseline
    mean_llr = group["fixed_llr"].mean()
    random_regret = mean_llr - min_llr
    
    per_shot_stats.append({
        "shot_id": shot_id,
        "spearman_rho": rho,
        "spearman_p": p_val,
        "min_llr_rank": min_llr_rank,
        "min_llr": min_llr,
        "lowest_rank": lowest_rank,
        "lowest_rank_llr": lowest_rank_llr,
        "lowest_rank_is_best": lowest_rank_is_best,
        "regret": regret,
        "random_regret": random_regret,
        "best_llr": best_llr,
        "mean_fixed_llr": mean_llr,
    })

df_per_shot = pd.DataFrame(per_shot_stats)

print("=" * 60)
print("PER-SHOT STATISTICS COMPUTED")
print("=" * 60)
print(f"\nAnalyzed {len(df_per_shot)} shots")
print(f"Errors per shot: {df_results.groupby('shot_id').size().iloc[0]}")
print(f"\nDataFrame preview:")
df_per_shot.head(10)

## 4. Per-Shot Spearman Correlation Statistics

In [ ]:
# ==============================================================================
# Per-Shot Correlation Statistics
# ==============================================================================

print("=" * 60)
print("PER-SHOT SPEARMAN CORRELATION STATISTICS")
print("=" * 60)

# Filter out NaN correlations (can happen if all LLRs are identical in a shot)
valid_correlations = df_per_shot["spearman_rho"].dropna()
n_valid = len(valid_correlations)
n_total = len(df_per_shot)

print(f"\nValid correlations: {n_valid}/{n_total} shots")

# Basic statistics
mean_rho = valid_correlations.mean()
std_rho = valid_correlations.std()
median_rho = valid_correlations.median()

print(f"\nPer-shot Spearman correlation (rank vs LLR):")
print(f"  Mean:   {mean_rho:.4f}")
print(f"  Std:    {std_rho:.4f}")
print(f"  Median: {median_rho:.4f}")
print(f"  Min:    {valid_correlations.min():.4f}")
print(f"  Max:    {valid_correlations.max():.4f}")

# One-sample t-test: is mean correlation significantly different from 0?
t_stat, t_pval = stats.ttest_1samp(valid_correlations, 0)
print(f"\nOne-sample t-test (H0: mean = 0):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {t_pval:.2e}")
print(f"  Interpretation: {'Mean is significantly different from 0' if t_pval < 0.05 else 'Mean is NOT significantly different from 0'}")

# Wilcoxon signed-rank test (non-parametric alternative)
w_stat, w_pval = stats.wilcoxon(valid_correlations)
print(f"\nWilcoxon signed-rank test (H0: median = 0):")
print(f"  W-statistic: {w_stat:.1f}")
print(f"  p-value: {w_pval:.2e}")

# Fraction of positive/negative correlations
n_positive = (valid_correlations > 0).sum()
n_negative = (valid_correlations < 0).sum()
n_zero = (valid_correlations == 0).sum()
print(f"\nCorrelation sign distribution:")
print(f"  Positive (rank ↑ → LLR ↑): {n_positive} ({100*n_positive/n_valid:.1f}%)")
print(f"  Negative (rank ↑ → LLR ↓): {n_negative} ({100*n_negative/n_valid:.1f}%)")
print(f"  Zero:                       {n_zero} ({100*n_zero/n_valid:.1f}%)")

In [ ]:
# ==============================================================================
# Histogram of Per-Shot Correlations
# ==============================================================================

fig = go.Figure()

# Histogram of per-shot Spearman correlations
fig.add_trace(go.Histogram(
    x=valid_correlations,
    nbinsx=50,
    name="Per-shot ρ",
    marker_color="steelblue",
    opacity=0.7,
))

# Add vertical line at mean
fig.add_vline(
    x=mean_rho,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean = {mean_rho:.3f}",
    annotation_position="top",
)

# Add vertical line at 0
fig.add_vline(
    x=0,
    line_dash="solid",
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title="Distribution of Per-Shot Spearman Correlations (Rank vs LLR)",
    xaxis_title="Spearman ρ (per shot)",
    yaxis_title="Count",
    showlegend=False,
)

# Add annotation with statistics
fig.add_annotation(
    text=f"Mean = {mean_rho:.3f}<br>Std = {std_rho:.3f}<br>t-test p = {t_pval:.2e}",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=11),
    bgcolor="white",
    bordercolor="gray",
    borderwidth=1,
    align="left",
)

fig.show()

## 5. Optimal Selection Rate & Regret Analysis

In [ ]:
# ==============================================================================
# Optimal Selection Rate and Regret Analysis
# ==============================================================================

print("=" * 60)
print("OPTIMAL SELECTION RATE")
print("=" * 60)

n_errors = len(df_results["error_rank"].unique())
random_baseline = 1.0 / n_errors
lowest_rank = df_per_shot["lowest_rank"].iloc[0]

# Lowest-rank selection success rate
lowest_rank_success_rate = df_per_shot["lowest_rank_is_best"].mean()
print(f"\nIf we always select rank-{lowest_rank} (highest probability) error:")
print(f"  Success rate (rank-{lowest_rank} has min LLR): {100*lowest_rank_success_rate:.2f}%")
print(f"  Random baseline:                   {100*random_baseline:.2f}%")
print(f"  Improvement over random:           {100*(lowest_rank_success_rate - random_baseline):.2f}pp")

# Binomial test: is success rate better than random?
binom_result = stats.binomtest(
    k=int(df_per_shot["lowest_rank_is_best"].sum()),
    n=len(df_per_shot),
    p=random_baseline,
    alternative="greater",
)
print(f"\nBinomial test (H0: success rate = random):")
print(f"  p-value: {binom_result.pvalue:.2e}")
print(f"  95% CI: [{binom_result.proportion_ci(confidence_level=0.95).low:.4f}, {binom_result.proportion_ci(confidence_level=0.95).high:.4f}]")

print("\n" + "=" * 60)
print("REGRET ANALYSIS")
print("=" * 60)

mean_regret = df_per_shot["regret"].mean()
mean_random_regret = df_per_shot["random_regret"].mean()

print(f"\nRegret = LLR(selected) - LLR(optimal)")
print(f"\nLowest-rank (rank-{lowest_rank}) selection regret:")
print(f"  Mean:   {mean_regret:.2f}")
print(f"  Median: {df_per_shot['regret'].median():.2f}")
print(f"  Std:    {df_per_shot['regret'].std():.2f}")

print(f"\nRandom selection regret (expected):")
print(f"  Mean:   {mean_random_regret:.2f}")
print(f"  Median: {df_per_shot['random_regret'].median():.2f}")
print(f"  Std:    {df_per_shot['random_regret'].std():.2f}")

print(f"\nRegret reduction (random → rank-{lowest_rank}): {mean_random_regret - mean_regret:.2f}")

# Paired t-test: is lowest-rank regret lower than random regret?
t_regret, p_regret = stats.ttest_rel(df_per_shot["regret"], df_per_shot["random_regret"])
print(f"\nPaired t-test (H0: rank-{lowest_rank} regret = random regret):")
print(f"  t-statistic: {t_regret:.4f}")
print(f"  p-value: {p_regret:.2e}")

## 6. Rank-Stratified Success Rate (Top-k Analysis)

In [ ]:
# ==============================================================================
# Rank-Stratified Success Rate (Top-k Analysis)
# ==============================================================================

print("=" * 60)
print("RANK-STRATIFIED SUCCESS RATE")
print("=" * 60)
print("\nIf we pick among top-k errors by distribution probability,")
print("how often does the true minimum LLR fall within that set?")
print()

unique_ranks = sorted(df_results["error_rank"].unique())
n_total_ranks = len(unique_ranks)

topk_results = []
for k in range(1, n_total_ranks + 1):
    top_k_ranks = set(unique_ranks[:k])
    success_count = (df_per_shot["min_llr_rank"].isin(top_k_ranks)).sum()
    success_rate = success_count / len(df_per_shot)
    random_rate = k / n_total_ranks
    topk_results.append({
        "k": k,
        "ranks": f"{unique_ranks[0]}-{unique_ranks[k-1]}",
        "success_rate": success_rate,
        "random_rate": random_rate,
        "improvement": success_rate - random_rate,
    })

df_topk = pd.DataFrame(topk_results)
print(df_topk.to_string(index=False, formatters={
    "success_rate": "{:.2%}".format,
    "random_rate": "{:.2%}".format,
    "improvement": "{:+.2%}".format,
}))

In [ ]:
# ==============================================================================
# Visualization: Top-k Success Rate
# ==============================================================================

fig = go.Figure()

# Actual success rate
fig.add_trace(go.Scatter(
    x=df_topk["k"],
    y=df_topk["success_rate"],
    mode="lines+markers",
    name="Actual",
    line=dict(color="steelblue", width=2),
    marker=dict(size=8),
))

# Random baseline
fig.add_trace(go.Scatter(
    x=df_topk["k"],
    y=df_topk["random_rate"],
    mode="lines",
    name="Random baseline",
    line=dict(color="red", width=2, dash="dash"),
))

fig.update_layout(
    title="Top-k Success Rate: Does Distribution Rank Predict Winner?",
    xaxis_title="k (number of top ranks considered)",
    yaxis_title="Success Rate (min LLR in top-k)",
    yaxis_tickformat=".0%",
    legend=dict(x=0.7, y=0.1),
)

fig.show()

## 7. Distribution of Winning Rank

In [ ]:
# ==============================================================================
# Distribution of Winning Rank
# ==============================================================================

print("=" * 60)
print("DISTRIBUTION OF WINNING RANK")
print("=" * 60)
print("\nWhich rank achieves the minimum LLR most often?")
print()

win_counts = df_per_shot["min_llr_rank"].value_counts().sort_index()
expected_pct = 100.0 / n_total_ranks

winner_df = pd.DataFrame({
    "rank": unique_ranks,
    "wins": [win_counts.get(r, 0) for r in unique_ranks],
    "win_pct": [win_counts.get(r, 0) / len(df_per_shot) * 100 for r in unique_ranks],
    "expected_pct": expected_pct,
})
winner_df["vs_expected"] = winner_df["win_pct"] - expected_pct

print(winner_df.to_string(index=False, formatters={
    "win_pct": "{:.1f}%".format,
    "expected_pct": "{:.1f}%".format,
    "vs_expected": "{:+.1f}pp".format,
}))

In [ ]:
# ==============================================================================
# Visualization: Winning Rank Distribution
# ==============================================================================

fig = go.Figure()

# Bar chart of win percentages
fig.add_trace(go.Bar(
    x=[str(r) for r in winner_df["rank"]],
    y=winner_df["win_pct"],
    name="Win %",
    marker_color="steelblue",
    text=[f"{p:.1f}%" for p in winner_df["win_pct"]],
    textposition="outside",
))

# Random baseline
fig.add_hline(
    y=expected_pct,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Random: {expected_pct:.1f}%",
    annotation_position="top right",
)

fig.update_layout(
    title="Which Rank Wins Most Often?",
    xaxis_title="Error Rank",
    yaxis_title="Win Percentage (%)",
    showlegend=False,
)

fig.show()

## 8. Conditional Analysis by Shot Difficulty

In [ ]:
# ==============================================================================
# Conditional Analysis by Shot Difficulty
# ==============================================================================

print("=" * 60)
print("CONDITIONAL ANALYSIS BY SHOT DIFFICULTY")
print("=" * 60)
print("\nDoes correlation vary with shot difficulty (baseline LLR)?")

# Stratify by best_llr quartiles
df_per_shot["difficulty_quartile"] = pd.qcut(
    df_per_shot["best_llr"], 
    q=4, 
    labels=["Q1 (easy)", "Q2", "Q3", "Q4 (hard)"]
)

# Compute statistics per quartile
quartile_stats = df_per_shot.groupby("difficulty_quartile", observed=True).agg({
    "spearman_rho": ["mean", "std", "count"],
    "lowest_rank_is_best": "mean",
    "regret": "mean",
    "best_llr": ["min", "max"],
}).round(4)

quartile_stats.columns = [
    "rho_mean", "rho_std", "n_shots",
    "lowest_rank_success", "mean_regret",
    "llr_min", "llr_max"
]

print("\nPer-quartile statistics:")
print(quartile_stats.to_string())

# Test if correlation varies significantly across quartiles
groups_by_quartile = [
    group["spearman_rho"].dropna() 
    for _, group in df_per_shot.groupby("difficulty_quartile", observed=True)
]
kw_stat, kw_pval = stats.kruskal(*groups_by_quartile)
print(f"\nKruskal-Wallis test (correlation differs across difficulty quartiles):")
print(f"  H-statistic: {kw_stat:.4f}")
print(f"  p-value: {kw_pval:.2e}")

In [ ]:
# ==============================================================================
# Visualization: Per-Shot Analysis by Difficulty
# ==============================================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Per-Shot Correlation vs Baseline LLR",
        "Lowest-Rank Success Rate by Difficulty"
    ),
)

# Left: Scatter plot of correlation vs baseline LLR
fig.add_trace(
    go.Scatter(
        x=df_per_shot["best_llr"],
        y=df_per_shot["spearman_rho"],
        mode="markers",
        marker=dict(size=4, opacity=0.3, color="steelblue"),
        name="Shots",
        hovertemplate="Baseline LLR: %{x:.1f}<br>Correlation: %{y:.3f}<extra></extra>",
    ),
    row=1, col=1,
)

# Add horizontal line at y=0
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

# Right: Bar plot of success rate by quartile
quartile_labels = quartile_stats.index.tolist()
success_rates = quartile_stats["lowest_rank_success"].values

fig.add_trace(
    go.Bar(
        x=quartile_labels,
        y=success_rates,
        name="Success Rate",
        marker_color="steelblue",
        text=[f"{r:.1%}" for r in success_rates],
        textposition="outside",
    ),
    row=1, col=2,
)

# Add random baseline
fig.add_hline(
    y=random_baseline, 
    line_dash="dash", 
    line_color="red", 
    row=1, col=2,
    annotation_text=f"Random: {random_baseline:.1%}",
    annotation_position="top right",
)

fig.update_xaxes(title_text="Baseline LLR (shot difficulty)", row=1, col=1)
fig.update_yaxes(title_text="Per-Shot Spearman ρ", row=1, col=1)
fig.update_xaxes(title_text="Difficulty Quartile", row=1, col=2)
fig.update_yaxes(title_text="Lowest-Rank Success Rate", row=1, col=2)

fig.update_layout(
    height=400,
    showlegend=False,
    title_text="Per-Shot Analysis by Shot Difficulty",
)

fig.show()

## 9. Summary & Conclusions

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

# Rank range info
rank_range = metadata.get("rank_range")
if rank_range:
    print(f"\nRank range: [{rank_range[0]}, {rank_range[1]}) step {rank_range[2]}")
print(f"Selected {n_total_ranks} errors with ranks: {unique_ranks[0]} to {unique_ranks[-1]}")

print(f"\n--- Per-Shot Correlation ---")
print(f"Mean Spearman ρ: {mean_rho:.4f} ± {std_rho:.4f}")
print(f"Significantly different from 0: {'Yes' if t_pval < 0.05 else 'No'} (p = {t_pval:.2e})")

print(f"\n--- Lowest-Rank Selection (rank {lowest_rank}) ---")
print(f"Success rate: {100*lowest_rank_success_rate:.2f}%")
print(f"Random baseline: {100*random_baseline:.2f}%")
print(f"Improvement: {100*(lowest_rank_success_rate - random_baseline):+.2f}pp")

print(f"\n--- Regret ---")
print(f"Mean regret (rank-{lowest_rank}): {mean_regret:.2f}")
print(f"Mean regret (random): {mean_random_regret:.2f}")
print(f"Regret reduction: {mean_random_regret - mean_regret:.2f}")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

# Interpret based on results
if abs(mean_rho) < 0.1:
    print("\n• Per-shot correlation is NEGLIGIBLE (|ρ| < 0.1)")
    print("  → Distribution rank does NOT predict LLR ordering within shots")
elif mean_rho > 0.1:
    print("\n• Per-shot correlation is POSITIVE")
    print("  → Higher rank tends to have higher LLR (as expected)")
else:
    print("\n• Per-shot correlation is NEGATIVE (unexpected)")
    print("  → Higher rank tends to have LOWER LLR")

if lowest_rank_success_rate > random_baseline * 1.5:
    print(f"\n• Lowest-rank selection has STRONG advantage ({lowest_rank_success_rate/random_baseline:.1f}x random)")
elif lowest_rank_success_rate > random_baseline * 1.1:
    print(f"\n• Lowest-rank selection has MODERATE advantage ({lowest_rank_success_rate/random_baseline:.1f}x random)")
else:
    print(f"\n• Lowest-rank selection has MINIMAL advantage ({lowest_rank_success_rate/random_baseline:.1f}x random)")